Joyce Wang

02/17/2026

# Gene Set Enrichment Analysis with GSEApy: Head-to-Head Comparisons

In [1]:
# Core libraries
import hisepy
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
import seaborn as sns

from gseapy import dotplot

/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/VertexPartition.py:413: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\frac{1}{m} \\sum_{ij} \\left(A_{ij} - \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/VertexPartition.py:788: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\sum_{ij} \\left(A_{ij} - \\gamma \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/Optimiser.py:27: SyntaxWarning: invalid escape sequence '\g'
  implementation therefore does not guarantee subpartition :math:`\gamma`-density.
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/Optimiser.py:346: SyntaxWarning: invalid escape sequence '\s'
  .. math:: Q = \sum_k \\lambda_k Q_k.


# Load in DEG Results for head-to-head comparisons between formulations for each cell type

In [2]:
file_id = ['5a715df8-b774-4947-a023-5fd9c9d97e19']

# dict
file_path = hisepy.read_files(file_list=file_id)

In [3]:
# dict values --> list
deg_dfs = list(file_path.values())

In [4]:
# concatenate
deg_df = pd.concat(deg_dfs, ignore_index=True)

deg_df

,projectGuid,mergeKey,emr,labLastModified,lastUpdated,surveyLastModified,surveyScheme,cohort.cohortGuid,file.id,file.name,...,Unnamed: 0,names,scores,logfoldchanges,pvals,pvals_adj,cell_type,form_1,form_2,filename
0,910ae58c-41c3-49fb-8e48-08fb25529a61,5a715df8-b774-4947-a023-5fd9c9d97e19_00000000-...,,0001-01-01T00:00:00Z,2026-02-15T08:10:51.087Z,0001-01-01T00:00:00Z,,,5a715df8-b774-4947-a023-5fd9c9d97e19,/home/workspace/input/3733913762/2025_bgmp/5a7...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,CD44,9.302001,0.051674,1.378264e-20,1.842739e-17,CD4 Central Memory,Afatinib,Afatinib dimaleate,/home/workspace/input/3733913762/2025_bgmp/5a7...
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,CD2,8.935438,0.094797,4.055687e-19,2.711227e-16,CD4 Central Memory,Afatinib,Afatinib dimaleate,/home/workspace/input/3733913762/2025_bgmp/5a7...
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,S100A10,7.404135,0.028103,1.320082e-13,5.673898e-11,CD4 Central Memory,Afatinib,Afatinib dimaleate,/home/workspace/input/3733913762/2025_bgmp/5a7...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83437,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1324.0,SELL,-2.734802,-0.350373,6.241792e-03,6.380187e-01,CD8 Effector Memory,Tofacitinib citrate,Tofacitinib,/home/workspace/input/3733913762/2025_bgmp/5a7...
83438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1325.0,AKT3,-2.773036,-0.285096,5.553597e-03,6.150609e-01,CD8 Effector Memory,Tofacitinib citrate,Tofacitinib,/home/workspace/input/3733913762/2025_bgmp/5a7...
83439,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1326.0,PRKCQ,-2.781397,-0.195473,5.412555e-03,6.150609e-01,CD8 Effector Memory,Tofacitinib citrate,Tofacitinib,/home/workspace/input/3733913762/2025_bgmp/5a7...
83440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1327.0,RIPOR2,-2.910409,-0.245898,3.609560e-03,6.150609e-01,CD8 Effector Memory,Tofacitinib citrate,Tofacitinib,/home/workspace/input/3733913762/2025_bgmp/5a7...


## Clean up the DEG dataframe

In [5]:
# Remove empty rows
deg_df = deg_df.dropna(subset=["names"])

In [6]:
# Keep only columns needed for GSEA
deg_df = deg_df[[
    "cell_type",
    "form_1",
    "form_2",
    "names",
    "logfoldchanges",
    "pvals",
    "pvals_adj"
]]

In [7]:
# Adding a comparison label column
deg_df["comparison"] = deg_df["form_1"] + "_vs._" + deg_df["form_2"]

# Gene set `KEGG_2019_Human` for Gene Set Enrichment Analysis

In [8]:
### Add a dict to store results
gsea_res = {}

for cell_type, cell_type_df in deg_df.groupby("cell_type"):
    ### Nested dict to store each drug
    gsea_res[cell_type] = {}

    for comparison, comparison_df in cell_type_df.groupby("comparison"):
        
        # Build a ranked gene list: extract ranked genes (names + logFC)
        gene_rank = comparison_df[["names", "logfoldchanges"]]

        # With GSEA of drug treatments vs. DMSO control, previously filtered genes expressed in at least 30 cells with `sc.pp.calculate_qc_metrics` and `subset_cell.var.n_cells_by_counts()`.
        
        # Build a ranked gene list: Sort the genes from high to low fold changes
        gene_rank = gene_rank.sort_values("logfoldchanges", ascending=False)
        
        # Run prerank GSEA with gp.prerank()
        pre_res = gp.prerank(
            rnk=gene_rank,
            gene_sets = "KEGG_2019_Human",
            # default min_size = 15 -- note, number of features is reduced already by limited gene panel
            outdir = None, # don't write to disk
            verbose = False # make True to see what's going on behind the scenes
        )

        ### Store the output in the dict
        gsea_res[cell_type][comparison] = pre_res

## View results for each cell type and comparison, in a table

In [9]:
# View dictionary key to see what comparisons were made
gsea_res["CD4 Naive"].keys()

dict_keys(['Afatinib_vs._Afatinib dimaleate', 'Baricitinib_vs._Baricitinib phosphate', 'Canertinib_vs._Canertinib dihydrochloride', 'Erlotinib_vs._Erlotinib hydrochloride', 'Gefitinib_vs._Gefitinib hydrochloride', 'NVP-BSK805_vs._NVP-BSK805 2HCl', 'Ruxolitinib_vs._Ruxolitinib phosphate', 'Tofacitinib citrate_vs._Tofacitinib'])

In [10]:
# View results for each cell type and comparison (8 total comparisons!)
pre_res = gsea_res['CD4 Naive']['Tofacitinib citrate_vs._Tofacitinib']
pre_res.res2d.sort_values('FDR q-val').head(10)

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
1,prerank,Antigen processing and presentation,-0.644897,-1.831371,0.006623,0.046414,0.075,5/19,8.23%,KLRC4;CD8A;KLRD1;TNF;LGMN
2,prerank,Oxytocin signaling pathway,-0.589639,-1.783941,0.0,0.054229,0.122,8/26,11.90%,FOS;RGS2;PIK3CG;CD38;MAP2K1;MAPK3;MAP2K5;JUN
0,prerank,"Parathyroid hormone synthesis, secretion and a...",-0.615835,-1.844123,0.006652,0.077516,0.066,7/21,9.82%,FOS;BCL2;FGFR1;MAP2K1;MAPK3;EGR1;RXRA
3,prerank,Hematopoietic cell lineage,-0.500973,-1.619274,0.013921,0.190918,0.377,7/32,8.55%,IL3RA;CD24;CD8A;TNF;ITGA5;CD38;CD3G
7,prerank,Cholinergic synapse,-0.502919,-1.487511,0.058957,0.307465,0.684,5/23,7.75%,FOS;BCL2;PIK3CG;MAP2K1;MAPK3
6,prerank,IL-17 signaling pathway,-0.471274,-1.496204,0.037313,0.33654,0.666,10/31,17.17%,FOS;TNF;CEBPB;FOSB;MAPK3;TBK1;JUN;IKBKG;CHUK;T...
5,prerank,Malaria,-0.553817,-1.497432,0.055427,0.399445,0.663,6/17,3.67%,KLRK1;KLRB1;CD40LG;CD40;TNF;LRP1
15,prerank,Cytokine-cytokine receptor interaction,-0.372206,-1.364397,0.080402,0.591297,0.891,16/62,13.02%,IL3RA;CCR4;CCL4;CCL5;TNFRSF9;CD40LG;IL10RA;CD4...
18,prerank,cGMP-PKG signaling pathway,-0.416916,-1.292062,0.131336,0.724246,0.954,5/28,11.02%,RGS2;PIK3CG;MAP2K1;MAPK3;ADRB2
17,prerank,Natural killer cell mediated cytotoxicity,-0.386219,-1.306616,0.072639,0.736027,0.946,10/46,9.27%,KLRK1;GZMB;KLRD1;TNF;MICA;MAP2K1;MAPK3;PRF1;PA...


# Create data frame, data frame to CSV, upload to HISE

In [11]:
# empty list
all_results = []

for cell_type in gsea_res:
    for comparison in gsea_res[cell_type]:
        res_df = gsea_res[cell_type][comparison].res2d  # results in rows with columns like Term, NES, FDR q-val, etc...

        # Add more columns for detail
        res_df["cell_type"] = cell_type
        res_df["comparison"] = comparison
        res_df["gene_set"] = "KEGG"   # Hallmark, KEGG, or Reactome

        # Add to dataframe
        all_results.append(res_df)

# Create a dataframe
final_gsea_df = pd.concat(all_results)

In [12]:
final_gsea_df

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes,cell_type,comparison,gene_set
0,prerank,Hematopoietic cell lineage,-0.507675,-1.470535,0.024865,1.0,0.792,9/33,14.14%,ITGAM;IL5RA;IL1R1;ITGA2;CD38;IL11RA;TNF;ITGA1;...,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,KEGG
1,prerank,Type II diabetes mellitus,-0.557602,-1.394473,0.069212,1.0,0.938,7/15,23.56%,TNF;MTOR;PIK3CA;IKBKB;MAPK9;HK1;PRKCE,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,KEGG
2,prerank,Cytosolic DNA-sensing pathway,-0.519473,-1.343356,0.089431,1.0,0.985,6/18,20.19%,NFKBIB;IRF3;TBK1;IKBKE;POLR1C;IKBKB,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,KEGG
3,prerank,ErbB signaling pathway,-0.427066,-1.287956,0.093448,1.0,0.997,16/40,25.95%,BAD;PTK2;NCK2;CRK;NCK1;BRAF;HRAS;MTOR;PIK3CA;E...,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,KEGG
4,prerank,Adrenergic signaling in cardiomyocytes,-0.466955,-1.279362,0.143345,1.0,0.998,8/23,22.96%,ADRB2;TPM1;MAPK13;PIK3CG;ATF6B;PLCB3;PPP2R3C;AKT3,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,KEGG
...,...,...,...,...,...,...,...,...,...,...,...,...,...
138,prerank,C-type lectin receptor signaling pathway,-0.172744,-0.55487,0.986842,1.0,1.0,17/48,31.69%,FCER1G;EGR3;PIK3CA;NFATC1;MAPK11;IKBKE;JUN;CHU...,Treg,Tofacitinib citrate_vs._Tofacitinib,KEGG
139,prerank,Adipocytokine signaling pathway,-0.189601,-0.531733,0.988314,1.0,1.0,9/24,34.51%,NFKBIB;NFKBIE;CHUK;MTOR;TNF;MAPK8;IKBKG;PRKAG2...,Treg,Tofacitinib citrate_vs._Tofacitinib,KEGG
140,prerank,Fc gamma R-mediated phagocytosis,-0.190163,-0.530381,0.984211,1.0,1.0,3/23,14.82%,VAV3;PIK3CA;PRKCE,Treg,Tofacitinib citrate_vs._Tofacitinib,KEGG
141,prerank,Ubiquitin mediated proteolysis,-0.178998,-0.475943,0.994819,1.0,1.0,18/18,82.43%,KLHL9;UBE2J1;KEAP1;BIRC2;DDB2;UBE3A;UBE2A;MAP3...,Treg,Tofacitinib citrate_vs._Tofacitinib,KEGG


In [13]:
final_gsea_df.to_csv("il6_jak-stat_head_to_head_gsea_kegg.csv")

In [14]:
uuid_list = [
    '02765813-6130-4fac-8708-f01768aa05b6',
    '06b73fab-62d2-4fc3-8a86-2df960cbc1ea',
    '1aecccab-f62a-4c49-b2fe-ba5777930262',
    '207c5f6c-92ec-4690-a7fd-07b0cbebd9f6',
    '3adc31cf-1ea9-4a7d-bc63-31358cb8a325',
    '45e4bd43-4fae-49df-8974-8d043e395f73',
    '4c13d814-1493-48f8-8021-a819b556e97b',
    '56bc5070-2968-4c6b-8198-4e997c75e4fe',
    '64e7735c-e89b-41ef-a293-a39b9ed5ade9',
    '72755c82-880d-4814-8322-6a81ed07466d',
    '9693f71c-dcb3-4d40-b2ad-74fc2ad147de',
    '9f37360e-0191-418b-ab51-fdb86ab9be09',
    'a3bc4704-5fe3-4a74-bce5-0700689db1f6',
    'af42c180-0218-4918-ae12-0a4f43c4aa1f',
    'b790716c-b2de-4028-ad60-1e5a7b81f05d',
    'bfa4c2f1-69d5-4bcf-a1a6-9beb69dbffc9',
    'd076758f-3d76-49b7-91be-0217eefcdf26',
    'd2f8da49-85eb-4dea-a166-5308bb73de91'
]

file_list = hisepy.cache_files(uuid_list)

In [15]:
# a function to make unique destination strings using the periodic table elements
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
        
    rand_str = '-'.join(rand_el)
    return rand_str

In [16]:
# getting unique title for upload, using UTC time
from datetime import datetime, timezone

# getting the current time and saving it
utc_current = datetime.now(timezone.utc)
print(utc_current)

# listing the files to upload
files_to_upload = [
    "il6_jak-stat_head_to_head_gsea_kegg.csv"
]

# uploading the files
hisepy.upload_files(
    files = files_to_upload,
    study_space_id = "c8a94b84-b0b7-40a9-980b-81a63ad6e115",
    title = f"GSEApy_head-to-head_KEGG_data_h5ad_{utc_current}",
    input_file_ids = uuid_list,
    #destination is randomly generated elements
    destination = element_id()
)

2026-02-17 20:51:56.781498+00:00
checking if conda environment can compile...
creating temp conda environment...
temp conda environment created successfully, now packing...
Cannot determine the current notebook.
1) /home/workspace/gseapy_head_to_head_Hallmark_JW.ipynb
2) /home/workspace/gseapy_head_to_head_KEGG_JW.ipynb
3) /home/workspace/gseapy_KEGG_JW.ipynb
Please select (1-3) 


 2


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '7d0dfdda-a1fe-4a02-bccd-3575fad16830',
 'ProcessId': '58cca2e6-b274-4bf6-80db-6836b122b20e',
 'WorkflowId': 'a65cfdc2-9b42-4ef6-abff-24b16b114f10',
 'FileIds': ['cb5b75ed-e19a-4d5c-9fa0-956919361878']}